In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor

import optuna
from sklearn.model_selection import cross_val_score

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    explained_variance_score
)

from scipy.stats import pearsonr
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold



d:\Miniconda3\envs\PyTorchVirtualEnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
print('hello world')

hello world


In [3]:
data_df = pd.read_csv("D:\APSirCNTandLSTM\CNT with LSTM\Datasets\CNT_Reinforced Concrete_Final Dataset.csv") # Read the data

data_df.dropna(inplace = True) # Drop missing/null values

print("Original Data Shape: ", data_df.shape)

def is_numeric_row(row):
    try:
        pd.to_numeric(row, errors='raise')
        return True
    except:
        return False
    

numeric_rows = data_df.apply(lambda row: is_numeric_row(row), axis=1)
data_df = data_df[numeric_rows]

print("Data Shape (rows, cols): ", data_df.shape) # Print data shape
data_df # Print/visualize the first 5 rows of the data

Original Data Shape:  (295, 7)
Data Shape (rows, cols):  (295, 7)


,Cement (kg/m³),w/c,Coarse aggregate (kg/m³),Fine aggregate (kg/m³),Curing (days),CNT (%),Compressive strength(MPa)
0,360.0,0.50,950.0,745.0,28,0.00,39.500
1,320.0,0.56,898.0,782.0,28,0.00,40.000
2,328.5,0.68,533.7,625.1,28,0.00,40.000
3,360.0,0.50,898.0,797.0,28,0.00,40.000
4,320.0,0.56,845.0,834.0,28,0.00,40.300
...,...,...,...,...,...,...,...
290,400.0,0.45,987.0,780.0,28,0.05,64.000
291,450.0,0.35,1006.0,768.0,28,0.05,84.900
292,525.0,0.45,0.0,1580.0,28,0.00,29.875
293,525.0,0.45,0.0,1580.0,28,0.06,38.750


In [4]:
#Converting non-numeric columns to numeric, if any. This will convert non-numeric values to NaN, which will be dropped later.

for column in data_df.columns:
    data_df[column] = pd.to_numeric(data_df[column], errors='coerce')


target_column = "Compressive strength(MPa)"

# Check data types
print("Data types:")
print(data_df.dtypes)
print("\n" + "="*50 + "\n")

# Check if target column has issues
print(f"Target column '{target_column}' unique values (first 10):")
print(data_df[target_column].unique()[:10])
print(f"\nTarget column type: {data_df[target_column].dtype}")

# Check for any remaining object columns
object_cols = data_df.select_dtypes(include=['object']).columns.tolist()
print(f"\nObject (string) columns: {object_cols}")

Data types:
Cement (kg/m³)               float64
w/c                          float64
Coarse aggregate (kg/m³)     float64
Fine aggregate (kg/m³)       float64
Curing (days)                  int64
CNT (%)                      float64
Compressive strength(MPa)    float64
dtype: object


Target column 'Compressive strength(MPa)' unique values (first 10):
[39.5 40.  40.3 40.8 41.  41.1 41.3 41.5 41.8 42. ]

Target column type: float64

Object (string) columns: []


In [5]:
# Prepare Input and Output

X = np.array(data_df.iloc[:, :-1])
Y = np.array(data_df.iloc[:, -1])

print("Input Shape:", X.shape)
print("Output Shape:", Y.shape)


Input Shape: (295, 6)
Output Shape: (295,)


In [6]:
X, Y

(array([[3.600e+02, 5.000e-01, 9.500e+02, 7.450e+02, 2.800e+01, 0.000e+00],
        [3.200e+02, 5.600e-01, 8.980e+02, 7.820e+02, 2.800e+01, 0.000e+00],
        [3.285e+02, 6.800e-01, 5.337e+02, 6.251e+02, 2.800e+01, 0.000e+00],
        ...,
        [5.250e+02, 4.500e-01, 0.000e+00, 1.580e+03, 2.800e+01, 0.000e+00],
        [5.250e+02, 4.500e-01, 0.000e+00, 1.580e+03, 2.800e+01, 6.000e-02],
        [5.250e+02, 4.500e-01, 0.000e+00, 1.580e+03, 2.800e+01, 1.200e-01]]),
 array([39.5  , 40.   , 40.   , 40.   , 40.3  , 40.8  , 40.8  , 41.   ,
        41.1  , 41.3  , 41.3  , 41.5  , 41.5  , 41.5  , 41.5  , 41.8  ,
        42.   , 42.1  , 42.4  , 42.5  , 43.   , 43.2  , 43.3  , 43.5  ,
        43.6  , 43.8  , 44.   , 44.5  , 45.   , 45.   , 45.1  , 45.7  ,
        46.   , 46.   , 47.1  , 47.7  , 50.   , 36.8  , 43.1  , 38.6  ,
        47.2  , 45.1  , 49.6  , 47.4  , 54.   , 50.1  , 37.8  , 44.1  ,
        40.9  , 47.5  , 45.3  , 42.5  , 52.   , 48.7  , 46.6  , 54.5  ,
        53.1  , 49.2  , 4

In [7]:

# 80 : 20 

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.20, random_state=42
)

print("Training X:", X_train.shape)
print("Testing X:", X_test.shape)

print("Training y:", y_train.shape)
print("Testing y:", y_test.shape)


print("Training set is: ", X_train.shape[0], " rows which is ", round(X_train.shape[0]/data_df.shape[0],4)*100, "%") # Print training shape
print("Testing set is: ",X_test.shape[0], " rows which is ", round(X_test.shape[0]/data_df.shape[0],4)*100, "%") # Print testing shape


Training X: (236, 6)
Testing X: (59, 6)
Training y: (236,)
Testing y: (59,)
Training set is:  236  rows which is  80.0 %
Testing set is:  59  rows which is  20.0 %


In [8]:
scaler_X_BR = StandardScaler()

X_train_scaled = scaler_X_BR.fit_transform(X_train)
X_test_scaled = scaler_X_BR.transform(X_test)

print("Training Shape X:", X_train_scaled.shape)
print("testing Shape X:", X_test_scaled.shape)
print("**********************************")
print("Training Shape y:", y_train.shape)
print("testing Shape y:", y_test.shape)


Training Shape X: (236, 6)
testing Shape X: (59, 6)
**********************************
Training Shape y: (236,)
testing Shape y: (59,)


In [9]:

# Model

bag_model = BaggingRegressor(

    estimator=DecisionTreeRegressor(
        max_depth=4,
        random_state=42
    ),

    n_estimators=200,

    max_samples=0.8,

    max_features=1.0,

    bootstrap=True,

    random_state=42,

    n_jobs=-1
)

# Training

bag_model.fit(
    X_train_scaled,
    y_train
)

# Prediction

y_pred_train = bag_model.predict(X_train_scaled)
y_pred_test = bag_model.predict(X_test_scaled)



print(y_pred_test)
print(y_pred_train)


[51.6326642  53.80296544 48.97803021 48.97803021 40.26346184 41.32781695
 48.91023041 47.47468322 47.52467557 52.53785946 52.15612571 33.30314214
 47.28431972 47.6982772  47.69759459 45.06963078 48.77546896 37.41315937
 22.86480816 52.55811198 56.95836221 57.04653484 52.31069733 56.14711237
 44.13350627 41.18866115 37.34490367 20.62943481 44.37431    46.60033748
 27.72275049 44.7721477  22.15516365 22.15516365 36.98361669 45.08882775
 57.26455856 54.37690328 47.47468322 32.48857055 30.02621718 52.27741146
 50.12290803 51.93498203 47.31497841 41.11261978 49.16282172 45.21929457
 41.28755677 46.64772552 46.97141779 56.88688572 42.03396977 43.86679473
 19.06295209 44.00539306 48.77546896 46.51596918 54.37690328]
[49.48547922 43.75966536 47.00644535 46.77312402 22.86480816 57.00601927
 48.83185548 49.00426555 47.52467557 45.08882775 52.93240558 56.95836221
 48.15741932 42.11206369 51.77094077 46.57573931 55.97309699 43.46937628
 42.92490701 42.74290093 55.77386334 47.06555214 56.09941277 4

In [10]:
def evaluate_regression(y_true, y_pred):

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    r2 = r2_score(y_true, y_pred)

    mse = mean_squared_error(y_true, y_pred)

    rmse = np.sqrt(mse)

    mae = mean_absolute_error(y_true, y_pred)

    # avoid divide-by-zero
    mape = np.mean(
        np.abs((y_true - y_pred) / (y_true + 1e-8))
    ) * 100

    evs = explained_variance_score(y_true, y_pred)

    pearson_r, _ = pearsonr(y_true, y_pred)

    relative_rmse = (rmse / np.mean(y_true)) * 100

    metrics = {
        "R2": r2,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE (%)": mape,
        "Explained Variance Score": evs,
        "Pearson Correlation": pearson_r,
        "Relative RMSE (%)": relative_rmse
    }

    return metrics

print("TRAIN METRICS")
print(evaluate_regression(y_train, y_pred_train))


print("\nTEST METRICS")
print(evaluate_regression(y_test, y_pred_test))

TRAIN METRICS
{'R2': 0.8205338785827174, 'MSE': 19.347444168311373, 'RMSE': 4.3985729695335705, 'MAE': 3.1604017292168596, 'MAPE (%)': 7.775309309748855, 'Explained Variance Score': 0.8205380240652056, 'Pearson Correlation': 0.9186171201820182, 'Relative RMSE (%)': 9.63658654283376}

TEST METRICS
{'R2': 0.8523122469920646, 'MSE': 21.436793751299874, 'RMSE': 4.629988526044085, 'MAE': 3.211259379414408, 'MAPE (%)': 8.70466540803389, 'Explained Variance Score': 0.8526487642129327, 'Pearson Correlation': 0.9343867270731838, 'Relative RMSE (%)': 10.474163373757243}


In [11]:
import pandas as pd

train_metrics = evaluate_regression(y_train, y_pred_train)
test_metrics  = evaluate_regression(y_test, y_pred_test)

df_results = pd.DataFrame({
    "Train": train_metrics,
    "Test": test_metrics
})

print(df_results.round(4))

                            Train     Test
R2                         0.8205   0.8523
MSE                       19.3474  21.4368
RMSE                       4.3986   4.6300
MAE                        3.1604   3.2113
MAPE (%)                   7.7753   8.7047
Explained Variance Score   0.8205   0.8526
Pearson Correlation        0.9186   0.9344
Relative RMSE (%)          9.6366  10.4742


In [12]:
#Using Optuna 

# K-Fold
kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Objective function
def objective(trial):

    tree = DecisionTreeRegressor(

        max_depth=trial.suggest_categorical(
            "max_depth",
            [None, 3, 4, 5, 6, 8]
        ),

        min_samples_split=trial.suggest_categorical(
            "min_samples_split",
            [2, 5, 10]
        ),

        min_samples_leaf=trial.suggest_categorical(
            "min_samples_leaf",
            [1, 2, 4]
        ),

        random_state=42
    )

    model = BaggingRegressor(

        estimator=tree,

        n_estimators=trial.suggest_categorical(
            "n_estimators",
            [50, 100, 200, 300, 500]
        ),

        max_samples=trial.suggest_categorical(
            "max_samples",
            [0.5, 0.6, 0.7, 0.8, 1.0]
        ),

        max_features=trial.suggest_categorical(
            "max_features",
            [0.6, 0.8, 1.0]
        ),

        bootstrap=trial.suggest_categorical(
            "bootstrap",
            [True, False]
        ),

        random_state=42,
        n_jobs=-1
    )

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    scores = cross_val_score(
        pipeline,
        X_train,      # unscaled data
        y_train,
        cv=kfold,
        scoring="r2",
        n_jobs=-1
    )

    return scores.mean()

In [13]:
study = optuna.create_study(
    direction="maximize",
    study_name="BaggingRegressor Optimization"
)

study.optimize(
    objective,
    n_trials=50
)

[I 2026-06-03 15:00:23,930] A new study created in memory with name: BaggingRegressor Optimization
[I 2026-06-03 15:00:24,109] Trial 0 finished with value: 0.6289821816447596 and parameters: {'max_depth': 3, 'min_samples_split': 2, 'min_samples_leaf': 2, 'n_estimators': 100, 'max_samples': 0.8, 'max_features': 1.0, 'bootstrap': False}. Best is trial 0 with value: 0.6289821816447596.
[I 2026-06-03 15:00:24,414] Trial 1 finished with value: 0.6536065343238017 and parameters: {'max_depth': 4, 'min_samples_split': 10, 'min_samples_leaf': 2, 'n_estimators': 200, 'max_samples': 0.5, 'max_features': 0.6, 'bootstrap': False}. Best is trial 1 with value: 0.6536065343238017.
[I 2026-06-03 15:00:24,604] Trial 2 finished with value: 0.715309463985597 and parameters: {'max_depth': 6, 'min_samples_split': 5, 'min_samples_leaf': 2, 'n_estimators': 100, 'max_samples': 1.0, 'max_features': 0.6, 'bootstrap': True}. Best is trial 2 with value: 0.715309463985597.
[I 2026-06-03 15:00:24,772] Trial 3 finish

In [14]:
print("Best Parameters:")
print(study.best_params)

print("\nBest CV R2:")
print(study.best_value)

Best Parameters:
{'max_depth': None, 'min_samples_split': 5, 'min_samples_leaf': 1, 'n_estimators': 500, 'max_samples': 0.6, 'max_features': 1.0, 'bootstrap': False}

Best CV R2:
0.8287168908379943


In [15]:
best_params = study.best_params


best_tree = DecisionTreeRegressor(
    max_depth=best_params["max_depth"],
    min_samples_split=best_params["min_samples_split"],
    min_samples_leaf=best_params["min_samples_leaf"],
    random_state=42
)


best_bag = BaggingRegressor(
    estimator=best_tree,

    n_estimators=best_params["n_estimators"],

    max_samples=best_params["max_samples"],

    max_features=best_params["max_features"],

    bootstrap=best_params["bootstrap"],

    random_state=42,

    n_jobs=-1
)


best_bag.fit(
    X_train_scaled,
    y_train
)

,"estimator estimator: object, default=NoneThe base estimator to fit on random subsets of the dataset.If None, then the base estimator is a:class:`~sklearn.tree.DecisionTreeRegressor`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",DecisionTreeR...ndom_state=42)
,"n_estimators n_estimators: int, default=10The number of base estimators in the ensemble.",500
,"max_samples max_samples: int or float, default=NoneThe number of samples to draw from X to train each base estimator (withreplacement by default, see `bootstrap` for more details).- If None, then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples.",0.6
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator (without replacement by default, see `bootstrap_features` for moredetails).- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.",1.0
,"bootstrap bootstrap: bool, default=TrueWhether samples are drawn with replacement. If False, sampling withoutreplacement is performed. If fitting with `sample_weight`, it isstrongly recommended to choose True, as only drawing with replacementwill ensure the expected frequency semantics of `sample_weight`.",False
,"bootstrap_features bootstrap_features: bool, default=FalseWhether features are drawn with replacement.",False
,"oob_score oob_score: bool, default=FalseWhether to use out-of-bag samples to estimatethe generalization error. Only available if bootstrap=True.",False
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fita whole new ensemble. See :term:`the Glossary `.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for both :meth:`fit` and:meth:`predict`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random resampling of the original dataset(sample wise and feature wise).If the base estimator accepts a `random_state` attribute, a differentseed is generated for each instance in the ensemble.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",0


In [16]:
y_pred_train = best_bag.predict(X_train_scaled)

y_pred_test = best_bag.predict(X_test_scaled)

In [17]:

def evaluate_regression(y_true, y_pred):

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    r2 = r2_score(y_true, y_pred)

    mse = mean_squared_error(y_true, y_pred)

    rmse = np.sqrt(mse)

    mae = mean_absolute_error(y_true, y_pred)

    mape = np.mean(
        np.abs((y_true - y_pred) / (y_true + 1e-8))
    ) * 100

    evs = explained_variance_score(y_true, y_pred)

    pearson_r, _ = pearsonr(y_true, y_pred)

    relative_rmse = (rmse / np.mean(y_true)) * 100

    return {
        "R2": r2,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE (%)": mape,
        "Explained Variance Score": evs,
        "Pearson Correlation": pearson_r,
        "Relative RMSE (%)": relative_rmse
    }


train_metrics = evaluate_regression(y_train, y_pred_train)


test_metrics = evaluate_regression(y_test, y_pred_test)


df_results = pd.DataFrame({
    "Train": train_metrics,
    "Test": test_metrics
})


print(df_results.round(4))

                           Train     Test
R2                        0.9539   0.9264
MSE                       4.9748  10.6758
RMSE                      2.2304   3.2674
MAE                       1.4497   2.1373
MAPE (%)                  3.9231   6.2100
Explained Variance Score  0.9539   0.9281
Pearson Correlation       0.9788   0.9664
Relative RMSE (%)         4.8865   7.3916


In [18]:
#Save XGBoost model using pickle

import pickle

# save trained model
with open("D:\APSirCNTandLSTM\CNT with LSTM\practice_Finalized For LSTM-CNT Scripts\practice_LSTM and ANN_for publication\LSTM Improved\EnsemblesANNLSTM9010\BaggingRegressor\saved models BaggingRegressor\BagR_model_8020_updated.pkl", "wb") as f:
    pickle.dump(best_bag, f)

print("Bagging Regressor saved successfully.")

Bagging Regressor saved successfully.


In [19]:
import pickle

# save trained model
with open("D:\APSirCNTandLSTM\CNT with LSTM\practice_Finalized For LSTM-CNT Scripts\practice_LSTM and ANN_for publication\LSTM Improved\EnsemblesANNLSTM9010\BaggingRegressor\saved models BaggingRegressor\BagR_Scalar8020_updated.pkl", "wb") as f:
    pickle.dump(scaler_X_BR, f)

print("Bagging Regressor scalar saved successfully.")

Bagging Regressor scalar saved successfully.


In [21]:
#Load model back

import pickle

with open("D:\APSirCNTandLSTM\CNT with LSTM\practice_Finalized For LSTM-CNT Scripts\practice_LSTM and ANN_for publication\LSTM Improved\EnsemblesANNLSTM9010\BaggingRegressor\saved models BaggingRegressor\BagR_model_8020_updated.pkl", "rb") as f:
    loaded_bag = pickle.load(f)

print("Model loaded successfully.")

Model loaded successfully.


In [22]:
#Recompute predictions (train / val / test)

y_pred_train_loaded = loaded_bag.predict(X_train_scaled)
y_pred_test_loaded  = loaded_bag.predict(X_test_scaled)

In [24]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
    explained_variance_score
)

from scipy.stats import pearsonr


def evaluate(y_true, y_pred):

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    evs = explained_variance_score(y_true, y_pred)
    pearson_r, _ = pearsonr(y_true, y_pred)
    rrmse = (rmse / np.mean(y_true)) * 100

    return {
        "R2": r2,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE (%)": mape,
        "Explained Variance": evs,
        "Pearson Correlation": pearson_r,
        "Relative RMSE (%)": rrmse
    }

train_metrics_loaded = evaluate(y_train, y_pred_train_loaded)
test_metrics_loaded  = evaluate(y_test, y_pred_test_loaded)


results_df = pd.DataFrame({
    "Train": train_metrics_loaded,
    "Test": test_metrics_loaded
})

print(results_df.round(4))

                      Train     Test
R2                   0.9539   0.9264
MSE                  4.9748  10.6758
RMSE                 2.2304   3.2674
MAE                  1.4497   2.1373
MAPE (%)             3.9231   6.2100
Explained Variance   0.9539   0.9281
Pearson Correlation  0.9788   0.9664
Relative RMSE (%)    4.8865   7.3916
